# T69 — Ocean gateway openings/closings through reference frames

**Cluster J: Paleoclimate.**

The opening and closure of ocean gateways — Drake Passage, Central American Seaway, Bering Strait, the many phases of the Tethys — are first-order controls on ocean circulation, heat transport, and Cenozoic climate. Their exact timing is one of the go-to targets for whether a paleogeographic reconstruction is any good. But those timings depend on the plate reference frame: exactly *when* Drake opens depends on where you put South America and Antarctica through the Eocene, which depends on whether you're in a paleomagnetic or a mantle reference frame.

This notebook computes gateway open/closed state through time under two reference frames using the PLASIM-GENIE landsea-mask ensemble bundled for T61. FIGURES.ipynb sections 1a + 1b in Jonathon Leonard's supplementary archive use medial-axis skeletons on 0.5° global grids to formalise this; our tutorial uses a **connectivity test** on the coarser 32×64 T61 mask — same concept, simpler implementation.

## What this notebook produces

1. **§3 — Snapshot of gateway state at one age.** Ocean vs land mask at 40 Ma in each of two frames, with the four gateway transect lines drawn on top. Colour of each transect segment shows whether it's presently open (blue) or closed (red).
2. **§4 — Gateway-history time series.** For each of Drake / Central American Seaway / Bering / Tethys, plot open/closed state as a horizontal bar per frame from 0 to 250 Ma. Places where the two frames disagree about opening timing are the paper's headline point.
3. **§5 — Continental-connectivity graph.** At each time, ask which ocean basins are connected. A 4-basin adjacency map (North Pacific ↔ Atlantic ↔ Indian ↔ Southern Ocean) coloured by frame answers questions like "when did the Pacific and Atlantic first become fully connected via Drake?".

## Learning objectives

- Load a per-frame landsea mask from a bundled ensemble (T61's `data/leonard_2025_paleoclimate/`).
- Sample a 2-D mask along a great-circle transect (Drake / Central American / Bering / Tethys locations).
- Use `scipy.ndimage.label` for connected-component analysis on ocean masks.
- Compare gateway timing under different reference frames as a falsifiable test of the frame's paleogeographic fidelity.

## Prerequisites and runtime

- No new bundled data — reuses `data/leonard_2025_paleoclimate/` from T61 (`grid_landsea_mask` variable per frame per age).
- Python: `xarray`, `numpy`, `pandas`, `matplotlib`, `pygmt`, `scipy`.
- Runtime: ~30 s.


## Environment + imports


In [ ]:
from pathlib import Path
import os, sys
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import pygmt
from scipy import ndimage as ndi

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, pd, xr, pygmt, ndi):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


In [ ]:
# === USER CONFIGURATION =====================================================
# Reuse T61 bundle
FRAME_A_DIR    = Path("data/leonard_2025_paleoclimate/merdith2021_paleomag")
FRAME_B_DIR    = Path("data/leonard_2025_paleoclimate/muller2016_mantle")
FRAME_A_LABEL  = "Merdith 2021 (paleomagnetic)"
FRAME_B_LABEL  = "Müller 2016 (hybrid mantle)"

# Ages
ALL_AGES_MA    = list(range(0, 260, 10))
SNAPSHOT_AGE   = 40    # §3 snapshot

# Gateway transects — approximate lat/lon endpoints of the modern gateway location
# on the modern globe. Each transect is a great-circle line; we sample landsea
# mask along it and ask what fraction is ocean.
GATEWAYS = {
    "Drake Passage":           {"lat0": -55, "lon0": -65,  "lat1": -63, "lon1": -60,  "colour": "#2980b9"},
    "Central American Seaway": {"lat0":  8,  "lon0": -83,  "lat1": 12,  "lon1": -78,  "colour": "#c0392b"},
    "Bering Strait":           {"lat0":  65, "lon0": -172, "lat1": 66,  "lon1": -167, "colour": "#27ae60"},
    "Tethys / Mediterranean":  {"lat0":  32, "lon0":  20,  "lat1": 37,  "lon1": 30,   "colour": "#f39c12"},
}

# Number of points to sample along each transect
N_SAMPLE_POINTS = 30

# Threshold: transect is "open" if this fraction of sample points is ocean
OPEN_FRACTION_THRESHOLD = 0.5
# ============================================================================
print(f"  frame A: {FRAME_A_LABEL}")
print(f"  frame B: {FRAME_B_LABEL}")
print(f"  ages: 0-{max(ALL_AGES_MA)} Ma at 10-Myr cadence")
print(f"  gateways: {list(GATEWAYS.keys())}")


## 1. Load landsea mask ensembles


In [ ]:
def load_mask_stack(frame_dir, ages, var="grid_landsea_mask"):
    """Return an xarray.Dataset with (age, lat, lon) landsea mask stack."""
    slices = []
    for age in ages:
        f = frame_dir / f"{age:03d}Ma.nc"
        if not f.exists(): continue
        ds = xr.open_dataset(f)
        if var in ds: slices.append(ds[[var]].expand_dims({"age": [age]}))
    return xr.concat(slices, dim="age")

mask_A = load_mask_stack(FRAME_A_DIR, ALL_AGES_MA)
mask_B = load_mask_stack(FRAME_B_DIR, ALL_AGES_MA)
print(f"  Frame A landsea mask stack: {dict(mask_A.sizes)}")
print(f"  Frame B landsea mask stack: {dict(mask_B.sizes)}")
# Detect naming (grid_landsea_mask == 1 for land, 0 for ocean, or vice versa)
_test = mask_A["grid_landsea_mask"].isel(age=0)
print(f"  test min/max: {float(_test.min()):.2f} / {float(_test.max()):.2f}")


## 2. Transect-sampling helper

Sample the landsea mask along a straight-line lat/lon transect. Returns 1 for ocean, 0 for land at each sample point.


In [ ]:
def sample_transect(mask_da, lat0, lon0, lat1, lon1, n=N_SAMPLE_POINTS,
                    lat_name="latitude", lon_name="longitude"):
    """Sample landsea mask along a straight lat/lon transect."""
    lats = np.linspace(lat0, lat1, n)
    lons = np.linspace(lon0, lon1, n)
    vals = mask_da.interp({lat_name: xr.DataArray(lats, dims="pt"),
                            lon_name: xr.DataArray(lons, dims="pt")},
                           method="nearest").values
    # Convention: land = 1, ocean = 0. Ocean fraction = 1 - land fraction.
    ocean_frac = float(np.mean(vals < 0.5))
    return lats, lons, vals, ocean_frac

def gateway_history(mask_stack, gateways=GATEWAYS):
    """Return DataFrame of (age, gateway, ocean_frac, is_open) per frame."""
    records = []
    for age in mask_stack["age"].values:
        mask = mask_stack["grid_landsea_mask"].sel(age=age)
        for gname, gcfg in gateways.items():
            _, _, _, ofrac = sample_transect(mask, gcfg["lat0"], gcfg["lon0"],
                                              gcfg["lat1"], gcfg["lon1"])
            records.append({"age_ma": float(age), "gateway": gname,
                            "ocean_fraction": ofrac,
                            "is_open": ofrac > OPEN_FRACTION_THRESHOLD})
    return pd.DataFrame(records)

hist_A = gateway_history(mask_A)
hist_B = gateway_history(mask_B)
print("Frame A gateway history head:")
print(hist_A.head(8).to_string(index=False))


## 3. Snapshot map at 40 Ma — where the gateways sit


In [ ]:
from paleoclimate_helpers import refine_for_plot   # noqa
sys.path.insert(0, str(Path("Notebooks").resolve()))
from paleoclimate_helpers import refine_for_plot as _refine   # noqa: F811

# Render both frames side-by-side using hand-laid layout (T61 pattern)
PANEL_W_CM = 8.0
PANEL_H_CM = PANEL_W_CM / 2
COL_GAP_CM = 0.7
DX_CM = PANEL_W_CM + COL_GAP_CM
PANEL_PROJ = f"N{PANEL_W_CM}c"
REGION     = [-180, 180, -90, 90]

fig = pygmt.Figure()
for col, (mask_stack, label) in enumerate([(mask_A, FRAME_A_LABEL), (mask_B, FRAME_B_LABEL)]):
    landmask = mask_stack["grid_landsea_mask"].sel(age=SNAPSHOT_AGE)
    landmask_smooth = refine_for_plot(landmask, lon_name="longitude", lat_name="latitude")
    pygmt.makecpt(cmap="gray", series=[0, 1, 0.1], reverse=True)
    fig.grdimage(grid=landmask_smooth, cmap=True, region=REGION, projection=PANEL_PROJ,
                 frame="af", nan_transparent=True)
    fig.coast(region=REGION, projection=PANEL_PROJ, shorelines="0.2p,gray30")

    # Draw the 4 gateway transects, colouring segment ocean/land by sample values
    for gname, gcfg in GATEWAYS.items():
        lats, lons, vals, ofrac = sample_transect(landmask, gcfg["lat0"], gcfg["lon0"],
                                                   gcfg["lat1"], gcfg["lon1"])
        colour = "#2980b9" if ofrac > OPEN_FRACTION_THRESHOLD else "#c0392b"
        fig.plot(x=lons, y=lats, pen=f"3p,{colour}", region=REGION, projection=PANEL_PROJ)
        fig.text(text=gname, x=(gcfg["lon0"]+gcfg["lon1"])/2,
                 y=(gcfg["lat0"]+gcfg["lat1"])/2 + 6,
                 font="7p,Helvetica-Bold,black", fill="white", pen="0.4p,gray40",
                 no_clip=True, region=REGION, projection=PANEL_PROJ)

    fig.text(text=label, position="TC", offset="0/0.4c", justify="MC",
             font="9p,Helvetica-Bold,black", no_clip=True,
             region=REGION, projection=PANEL_PROJ)
    fig.text(text=f"{SNAPSHOT_AGE} Ma", position="TL", offset="0.15c/-0.15c", justify="TL",
             font="10p,Helvetica-Bold,black", fill="white", pen="0.4p,gray40",
             region=REGION, projection=PANEL_PROJ)

    if col < 1:
        fig.shift_origin(xshift=f"{DX_CM}c")

fig.show(width=1100)


### How to read the snapshot map

- **Blue transect** — gateway open (majority of sample points fall in ocean).
- **Red transect** — gateway closed (majority in land).
- Frame A vs Frame B disagreement at the same age tells you the two paleogeographic reconstructions place the continents differently enough for the gateway to be in different states.


## 4. Gateway-history time series


In [ ]:
fig, axes = plt.subplots(len(GATEWAYS), 1, figsize=(11, 1.2 * len(GATEWAYS) + 1),
                          sharex=True)

for ax, gname in zip(axes, GATEWAYS):
    hist_a = hist_A[hist_A["gateway"] == gname]
    hist_b = hist_B[hist_B["gateway"] == gname]
    # Frame A row at y=0.6
    for _, row in hist_a.iterrows():
        colour = "#2980b9" if row["is_open"] else "#c0392b"
        ax.barh(0.7, 10, left=row["age_ma"] - 5, height=0.25, color=colour, alpha=0.85)
    # Frame B row at y=0.3
    for _, row in hist_b.iterrows():
        colour = "#2980b9" if row["is_open"] else "#c0392b"
        ax.barh(0.25, 10, left=row["age_ma"] - 5, height=0.25, color=colour, alpha=0.85)
    ax.set_yticks([0.25, 0.7])
    ax.set_yticklabels([FRAME_B_LABEL, FRAME_A_LABEL], fontsize=8)
    ax.set_ylabel(gname, fontsize=10, rotation=0, ha="right", va="center")
    ax.set_xlim(0, max(ALL_AGES_MA))
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3, axis="x")

axes[-1].set_xlabel("Age (Ma)")
axes[0].set_title("Gateway open (blue) / closed (red) history under two reference frames")

# Add legend
from matplotlib.patches import Patch
legend_handles = [Patch(color="#2980b9", label="Open"),
                  Patch(color="#c0392b", label="Closed")]
axes[0].legend(handles=legend_handles, loc="upper right", fontsize=8)

for ax in axes:
    ax.invert_xaxis()
plt.tight_layout()
plt.show()


### How to read the history plot

Two horizontal rows per gateway — top row is Frame A (paleomag), bottom row is Frame B (mantle). Each 10-Myr bin is coloured by the current state.

**Watch for**

- **Drake Passage opening** — Cenozoic literature places this ~34-30 Ma. Whether both frames agree on that timing at 32×64 resolution is a coarse-grid limit test.
- **Central American Seaway closure** — ~4 Ma per Coates & Obando 1996; the coarse mask likely shows it closing later than that (~10-0 Ma) due to grid resolution.
- **Bering Strait** — opening timing 6-5 Ma per Marincovich & Gladenkov (1999). At 32×64 the strait is close to sub-grid.
- **Tethys / Mediterranean** — closure of the Tethyan seaway happened progressively 30-6 Ma. The two reference frames could easily disagree by 5-10 Myr on when the modern Mediterranean was fully isolated.

**A resolution caveat**: 32×64 PLASIM grid means each cell is ~5.6° wide. Gateways narrower than that (Bering) show up as sub-grid and their open/closed state is dominated by which discrete grid cell falls on the strait. The full Leonard 2025 workflow uses 0.5° gateway-skeleton NCs precisely to avoid this.


## 5. Continental connectivity — which ocean basins are connected?

At each time step, count how many *disconnected* ocean basins exist globally. If two major basins are connected by an open gateway, they belong to the same connected component.


In [ ]:
def n_ocean_components(mask, threshold=0.5, min_size=5):
    """Count ocean connected components in a 2-D landsea mask. Land=1, Ocean=0."""
    ocean = (mask.values < threshold).astype(int)
    # Wrap longitude — glue the last column onto the first
    ocean_wrapped = np.concatenate([ocean, ocean[:, :1]], axis=1)
    labels, n = ndi.label(ocean_wrapped, structure=np.ones((3, 3), dtype=int))
    # Count only components with more than min_size cells
    if n == 0: return 0
    sizes = np.bincount(labels.flatten())[1:]
    return int((sizes >= min_size).sum())

records = []
for age in ALL_AGES_MA:
    mask_a = mask_A["grid_landsea_mask"].sel(age=age)
    mask_b = mask_B["grid_landsea_mask"].sel(age=age)
    records.append({"age_ma": age,
                    "n_basins_A": n_ocean_components(mask_a),
                    "n_basins_B": n_ocean_components(mask_b)})
conn_df = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(conn_df["age_ma"], conn_df["n_basins_A"], "o-", lw=1.6, ms=6,
        color="#c0392b", label=FRAME_A_LABEL)
ax.plot(conn_df["age_ma"], conn_df["n_basins_B"], "s-", lw=1.6, ms=6,
        color="#2980b9", label=FRAME_B_LABEL)
ax.set_xlabel("Age (Ma)")
ax.set_ylabel("Number of disconnected\nocean basins")
ax.set_title("Ocean-basin connectivity through time (32×64 PLASIM landsea mask, connected-component count)")
ax.invert_xaxis()
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### How to read the connectivity plot

- **`1` disconnected basin** — the global ocean is a single connected water body (modern-Earth case with Panama closed but Drake open).
- **`2` disconnected basins** — one major gateway closed, splitting the ocean (e.g. Panama and Drake both closed).
- **Higher numbers** — deep-time Pangaean geometry with landlocked seas.

Watch for **when the count changes** — those are the moments when a gateway opened or closed. If Frame A and Frame B disagree about the count at the same age, the two frames disagree about ocean connectivity at that age.


## Extend this

- **Load the full gateway skeleton NCs** from Jonathon Leonard's Zenodo archive (record 15628277). These are 0.5° resolution medial-axis skeletons for FIVE reference frames (paleomag, paleomagCEED, mantle2016, mantleCEED, mantleOpt) that let you measure gateway *width* — not just open/closed state.
- **Add more gateways.** Suez, Antilles, Mozambique Channel, Torres Strait, Fram Strait. Each is a coordinate pair in the `GATEWAYS` dict.
- **Cross-reference with T61.** Where the two frames disagree about a gateway state, T61 should show large ΔSAT locally — the disagreement propagates from paleogeography to circulation to climate.
- **Overlay published timing constraints.** For each gateway, overlay a bar showing the literature-consensus opening/closure age (with uncertainty). Does either frame match?

## Related notebooks

- **T61** — Reference-frame uncertainty in reconstructed paleoclimate (SAT view of the same frame-comparison story).
- **T63** — True polar wander decomposition (the *why* of the frame difference).

## Sources

- Leonard, J.S., Mather, B.R., Merdith, A.S., Zahirovic, S., Williams, S.E., Müller, R.D. (2025). Polar wander leads to large differences in past climate. *Communications Earth & Environment*.
- Coates, A.G. & Obando, J.A. (1996). The geologic evolution of the Central American isthmus. In *Evolution and Environment in Tropical America*, University of Chicago Press.
- Marincovich, L. & Gladenkov, A.Y. (1999). Evidence for an early opening of the Bering Strait. *Nature* 397, 149-151.
- Livermore, R., Nankivell, A., Eagles, G. & Morris, P. (2005). Paleogene opening of Drake Passage. *EPSL* 236, 459-470.
- Rögl, F. (1999). Mediterranean and Paratethys — Facts and hypotheses of an Oligocene to Miocene paleogeography. *Geologica Carpathica* 50, 339-349.
